In [ ]:
# Cell 1: Baseline training with managed run folders
from pathlib import Path
import json
import sys

import gymnasium as gym
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor

# Find project root so this notebook works from any current folder
start_dir = Path.cwd().resolve()
project_root = next((p for p in [start_dir, *start_dir.parents] if (p / "core").is_dir()), start_dir)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from core.train_manager import TrainManager

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

thresholds = [-195, -190, -180, -170, -160]
total_timesteps_cap = 600_000

class MultiThresholdCallback(BaseCallback):
    def __init__(self, thresholds=None, verbose=0):
        super().__init__(verbose)
        self.thresholds = thresholds or [-195, -190, -180, -170, -160]
        self.threshold_steps = {t: None for t in self.thresholds}

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            if "episode" in info:
                episode_reward = info["episode"]["r"]
                for threshold in self.thresholds:
                    if episode_reward >= threshold and self.threshold_steps[threshold] is None:
                        self.threshold_steps[threshold] = self.num_timesteps
                        print(f"Baseline reached threshold {threshold} at step {self.num_timesteps}")
        return True

# Create a new baseline run folder
manager = TrainManager(base_dir=project_root / "logs_and_results", algo_name="baseline")
paths = manager.get_run_paths(create=True)
run_id = paths["run_id"]
run_dir = paths["run_dir"]
print(f"Start baseline run_{run_id} at {run_dir}")

# Save training parameters for this run
manager.save_params(
    run_id,
    {
        "algo": "baseline",
        "thresholds": thresholds,
        "total_timesteps": total_timesteps_cap,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    },
)

env = gym.make("MountainCar-v0", render_mode="rgb_array")
env = Monitor(env, str(paths["monitor_csv"]))

model = PPO(
    policy="MlpPolicy",
    env=env,
    learning_rate=0.0003,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    verbose=1,
    tensorboard_log=str(paths["tensorboard_dir"]),
    device="cuda" if torch.cuda.is_available() else "cpu",
)

callback = MultiThresholdCallback(thresholds=thresholds)
print("Start baseline training for 600000 timesteps")
model.learn(
    total_timesteps=total_timesteps_cap,
    callback=callback,
    tb_log_name=f"ppo_baseline_run_{run_id}",
)

model_path = paths["models_dir"] / "baseline_multi_thresh"
model.save(str(model_path))
print(f"Saved model: {model_path}.zip")

baseline_results = {
    str(t): (s if s is not None else total_timesteps_cap)
    for t, s in callback.threshold_steps.items()
}
with open(paths["thresholds_json"], "w", encoding="utf-8") as f:
    json.dump(baseline_results, f, indent=2)
print(f"Saved threshold metrics: {paths['thresholds_json']}")

env.close()

CUDA available: True
GPU: NVIDIA GeForce RTX 3070 Laptop GPU
Device: cuda
Using cuda device
Wrapping the env in a DummyVecEnv.
Start baseline training for 600000 timesteps
Logging to logs_and_results\tensorboard_logs\ppo_baseline_multi_thresh_1


d:\CodeFiles\RL-Gymnasium-PACSPL602013-Final-Project\final_project_env\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 200      |
|    ep_rew_mean     | -200     |
| time/              |          |
|    fps             | 623      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 200         |
|    ep_rew_mean          | -200        |
| time/                   |             |
|    fps                  | 498         |
|    iterations           | 2           |
|    time_elapsed         | 8           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.004535624 |
|    clip_fraction        | 0           |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.1        |
|    explained_variance   | 0.000732    |
|    learning_rate        | 0.

In [ ]:
# Cell 2: Evaluate one baseline run and save video in run folder
from pathlib import Path
import sys
import imageio.v2 as imageio
from IPython.display import Video
import numpy as np
import gymnasium as gym
from stable_baselines3 import PPO

# Choose run id manually, or keep None to use latest run
SELECT_RUN_ID = None

start_dir = Path.cwd().resolve()
project_root = next((p for p in [start_dir, *start_dir.parents] if (p / "core").is_dir()), start_dir)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from core.train_manager import TrainManager

manager = TrainManager(base_dir=project_root / "logs_and_results", algo_name="baseline")
run_ids = manager.list_run_ids()
if not run_ids:
    raise FileNotFoundError("No baseline runs found.")

run_id = SELECT_RUN_ID if SELECT_RUN_ID is not None else run_ids[-1]
paths = manager.get_run_paths(run_id=run_id, create=False)
model_path = paths["models_dir"] / "baseline_multi_thresh.zip"
video_path = paths["videos_dir"] / "baseline_multi_thresh_demo.mp4"

if not model_path.exists():
    raise FileNotFoundError(f"Model file not found: {model_path}")

loaded_model = PPO.load(str(model_path))
video_env = gym.make("MountainCar-v0", render_mode="rgb_array")

best_frames = []
best_reward = -np.inf

print(f"Evaluate baseline run_{run_id} and collect best episode")
for i in range(5):
    current_frames = []
    total_reward = 0.0
    obs, info = video_env.reset()
    terminated = False
    truncated = False

    while not (terminated or truncated):
        current_frames.append(video_env.render())
        action, _ = loaded_model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = video_env.step(action)
        total_reward += reward

    print(f"Attempt {i + 1}: reward = {total_reward}")
    if total_reward > best_reward:
        best_reward = total_reward
        best_frames = current_frames

print(f"Best reward: {best_reward}")
imageio.mimsave(str(video_path), best_frames, fps=30)
video_env.close()
print(f"Saved video: {video_path}")
Video(str(video_path), embed=True)

Evaluate model and collect best episode
Attempt 1: reward = -114.0
Attempt 2: reward = -115.0
Attempt 3: reward = -113.0
Attempt 4: reward = -114.0
Attempt 5: reward = -114.0
Best reward: -113.0


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (600, 400) to (608, 400) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
